# Iterative Title Generation and Optimization

This notebook uses the trained numeric inference models and qualitative LLM analysis to iteratively improve video titles for specific channels.

**Export Path:** `/content/drive/MyDrive/numeric_inference_outputs/title_optimization_results_v2.json`  
**Type:** JSON (List of objects containing original metadata, best title, and full iteration history)

In [2]:
# Setup and Dependencies
# We only upgrade google-generativeai to avoid NumPy 2.0 conflicts in Colab
!pip install -q google-generativeai==0.8.3
!pip install -q sentence-transformers

import os
import json
import time
import numpy as np
import pandas as pd
import hashlib
from google.colab import drive, userdata
import google.generativeai as genai
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from googleapiclient.discovery import build
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
# youtube = build('youtube', 'v3', api_key=YOUTUBE_API_KEY)

MODEL_NAME = 'gemini-3.1-flash-lite'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
EVAL_DATA_PATH = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
LLM_RESULTS_PATH = os.path.join(BASE_PATH, 'llm_analysis_results.json')
EMBEDDING_CACHE_PATH = os.path.join(BASE_PATH, 'video_title_embeddings_latest.json')
TRAIN_DATA_PATH = os.path.join(BASE_PATH, 'train_structured_latest.json')
GENERATION_CACHE_PATH = os.path.join(BASE_PATH, 'title_generation_cache_v2.json')
FINAL_RESULTS_PATH = os.path.join(BASE_PATH, 'title_optimization_results_v2.json')
DESCRIPTION_CACHE_PATH = os.path.join(BASE_PATH, 'video_descriptions_cache.json')

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# Data Loading
with open(EVAL_DATA_PATH, 'r') as f:
    eval_dataset = json.load(f)

with open(LLM_RESULTS_PATH, 'r') as f:
    llm_analysis = json.load(f)

with open(EMBEDDING_CACHE_PATH, 'r') as f:
    embedding_cache = json.load(f)

with open(TRAIN_DATA_PATH, 'r') as f:
    train_data = json.load(f)

def load_cache(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache, path):
    with open(path, 'w') as f:
        json.dump(cache, f, indent=4)

generation_cache = load_cache(GENERATION_CACHE_PATH)
description_cache = load_cache(DESCRIPTION_CACHE_PATH)
print("Data and caches loaded successfully.")

Data and caches loaded successfully.


In [4]:
# Reconstruct global PCA from training data
all_train_embeddings = []
for channel in train_data:
    for video in channel['videos']:
        all_train_embeddings.append(embedding_cache[video['title']])

X_train = np.array(all_train_embeddings)
pca = PCA(n_components=15, random_state=42)
pca.fit(X_train)

print(f"Reconstructed PCA with {pca.n_components_} components.")

Reconstructed PCA with 15 components.


In [5]:
# Utility functions
def predict_views(titles, channel_id, channel_models_params):
    """Predict log-views for a list of titles."""
    embs = embedding_model.encode(titles)
    reduced = pca.transform(embs)
    params = channel_models_params[channel_id]
    intercept = params['intercept']
    coeffs = np.array(params['coefficients'])
    predictions = (reduced @ coeffs) + intercept
    return predictions.tolist()

def get_video_description(video_id):
    if video_id in description_cache:
        return description_cache[video_id]

    try:
        request = youtube.videos().list(part="snippet", id=video_id)
        response = request.execute()
        if response['items']:
            desc = response['items'][0]['snippet']['description']
            description_cache[video_id] = desc
            save_cache(description_cache, DESCRIPTION_CACHE_PATH)
            return desc
    except Exception as e:
        print(f"Error fetching description for {video_id}: {e}")
    return ""

def calculate_semantic_distance(text1, text2):
    """Calculate 1 - cosine similarity between two texts."""
    if not text1 or not text2:
        return 1.0
    embs = embedding_model.encode([text1, text2])
    sim = cosine_similarity([embs[0]], [embs[1]])[0][0]
    return float(1 - sim)

In [6]:
# Configuration for target channels
TARGET_CHANNELS_REQUESTED = ["20VC with Harry Stebbings", "A16z"]
target_channel_data = []
channel_models_params = {}

target_channels_lower = [c.lower() for c in TARGET_CHANNELS_REQUESTED]

for channel in eval_dataset:
    if channel['channel_name'].lower() in target_channels_lower:
        target_channel_data.append(channel)
        channel_models_params[channel['channel_id']] = {
            'coefficients': channel['model']['coefficients'],
            'intercept': channel['model']['intercept']
        }

print(f"Targeted {len(target_channel_data)} channels.")

Targeted 2 channels.


In [17]:
def get_gemini_response(prompt):
    prompt_hash = hashlib.md5(prompt.encode()).hexdigest()
    if prompt_hash in generation_cache:
        return generation_cache[prompt_hash]

    model = genai.GenerativeModel(MODEL_NAME)
    for _ in range(3):
        try:
            response = model.generate_content(prompt)
            text = response.text
            generation_cache[prompt_hash] = text
            save_cache(generation_cache, GENERATION_CACHE_PATH)
            time.sleep(2)
            return text
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                print("Quota exceeded. Sleeping for 60s...")
                time.sleep(60)
            elif "context" in str(e).lower() or "token" in str(e).lower():
                raise e # Handle long context error outside
            else:
                print(f"Error: {e}. Retrying...")
                time.sleep(5)
    return ""

def parse_titles(text, count=20):
    """Extract titles from LLM response."""
    lines = text.strip().split('\n')
    titles = []
    for line in lines:
        line = line.strip()
        if not line: continue
        if line[0].isdigit() or line[0] in ['-', '*']:
            parts = line.split('. ', 1) if '.' in line else line.split(' ', 1)
            if len(parts) > 1:
                titles.append(parts[1].strip().strip('"'))
            else:
                titles.append(line.strip('-* ').strip('"'))
        else:
            titles.append(line.strip('"'))
        if len(titles) == count: break
    return titles[:count]

In [18]:
# Iterative Optimization Loop
results = []
NUM_ITERATIONS = 10
TITLES_PER_ITER = 20
VIDEOS_PER_CHANNEL = 20

for channel in target_channel_data:
    cid = channel['channel_id']
    cname = channel['channel_name']

    global_desc = llm_analysis['global_performance_descriptions'].get(cid, "")
    dim_analysis = llm_analysis['channel_significant_dimension_analysis'].get(cid, "")

    test_vids = channel['test_videos'][:VIDEOS_PER_CHANNEL]

    for vid in test_vids:
        base_title = vid['title']
        base_pred = float(np.log1p(vid['actual_views']))
        video_id = vid['video_id']

        description = get_video_description(video_id)
        original_dist_to_desc = calculate_semantic_distance(base_title, description)

        print(f"Optimizing title for {cname}: '{base_title}'")
        history = []

        try:
            for i in range(NUM_ITERATIONS):
                print(f"  Iteration {i+1}...")
                best_so_far = max([base_pred] + [max([t['score'] for t in h['titles']]) for h in history]) if history else base_pred
                prompt = f"""You are an expert YouTube title strategist for the channel '{cname}'.
Channel success drivers:
{global_desc}

Semantic performance analysis:
{dim_analysis}

Original Title: {base_title}
Current best predicted performance (log-views): {best_so_far:.4f}
"""
                if history:
                    all_prev = []
                    for h in history:
                        for t_obj in h['titles']:
                            all_prev.append(t_obj)
                    all_prev.sort(key=lambda x: x['score'], reverse=True)

                    prompt += "Previous suggestions feedback:"
                    prompt += "Top performing suggestions:"
                    for t_obj in all_prev[:5]:
                        prompt += f"- {t_obj['text']} (Score: {t_obj['score']:.4f})"

                    prompt += "Lower performing suggestions (Avoid these patterns):"
                    for t_obj in all_prev[-5:]:
                        prompt += f"- {t_obj['text']} (Score: {t_obj['score']:.4f})"

                prompt += f"Task: Generate {TITLES_PER_ITER} new, improved titles for this video that will maximize views. Return ONLY the {TITLES_PER_ITER} titles, one per line."

                llm_text = get_gemini_response(prompt)
                new_titles = parse_titles(llm_text, count=TITLES_PER_ITER)

                if not new_titles:
                     print("    Error: No titles generated.")
                     continue

                preds = predict_views(new_titles, cid, channel_models_params)
                iter_titles = []
                for t, p in zip(new_titles, preds):
                    iter_titles.append({'text': t, 'score': p})

                history.append({'iteration': i + 1, 'titles': iter_titles})
                best_idx = np.argmax(preds)
                print(f"    Best this round: '{new_titles[best_idx]}' ({preds[best_idx] - base_pred:+.4f} log-views)")
        except Exception as e:
            if "context" in str(e).lower() or "token" in str(e).lower():
                print(f"    Long context error at iteration {len(history)+1}. Moving to next video.")
            else:
                print(f"    Unexpected error: {e}")
                raise e

        if not history: continue

        all_optimized = []
        for h in history:
            all_optimized.extend(h['titles'])
        all_optimized.sort(key=lambda x: x['score'], reverse=True)
        best_title_obj = all_optimized[0]

        # Grounding evaluation
        best_dist_to_desc = calculate_semantic_distance(best_title_obj['text'], description)

        results.append({
            'channel': cname,
            'video_id': video_id,
            'original_title': base_title,
            'original_score': base_pred,
            'original_dist_to_description': original_dist_to_desc,
            'best_optimized_title': best_title_obj['text'],
            'best_optimized_score': best_title_obj['score'],
            'best_dist_to_description': best_dist_to_desc,
            'improvement': best_title_obj['score'] - base_pred,
            'improvement_pct': (np.expm1(best_title_obj['score']) - np.expm1(base_pred)) / np.expm1(base_pred) * 100 if np.expm1(base_pred) > 0 else 0,
            'history': history
        })

print("Optimization complete.")

Error fetching description for ZYB1l3ML1v0: name 'youtube' is not defined
Optimizing title for a16z: 'Udio: From Text to Tune'
  Iteration 1...
    Best this round: 'Why AI Music is the Next Massive Creator Economy Shift' (+2.0176 log-views)
  Iteration 2...
    Best this round: 'Udio Founders: The Playbook for Building a Viral AI Unicorn' (+1.8766 log-views)
  Iteration 3...
    Best this round: 'Udio Founders: Scaling the AI Startup That’s Disrupting Global Media' (+2.3941 log-views)
  Iteration 4...
    Best this round: 'Why the Creator Economy Will Never Be the Same: Udio Founders on the AI Shift' (+2.3853 log-views)
  Iteration 5...
    Best this round: 'Udio Founders: Scaling the AI Startup That’s Disrupting Global Media' (+2.3941 log-views)
  Iteration 6...
    Best this round: 'Udio Founders: Why Our AI Startup is the Biggest Threat to Global Media Giants' (+2.6367 log-views)
  Iteration 7...
    Best this round: 'Udio: Why This AI Startup is Poised to Disrupt the Global Media 

In [19]:
# Export final results
with open(FINAL_RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Final results exported to {FINAL_RESULTS_PATH}")

Final results exported to /content/drive/MyDrive/numeric_inference_outputs/title_optimization_results_v2.json
